In [1]:
import pandas as pd
from pyspark.sql import SparkSession, Row
from decimal import Decimal
from pyspark.sql.types import StructType, StringType, StructField, IntegerType, DecimalType, ByteType

spark = SparkSession.builder \
      .appName("sales_csv_demo") \
      .master("local[2]") \
      .getOrCreate()
sc = spark.sparkContext

In [2]:
schema = StructType([
    StructField("ORDERNUMBER", IntegerType(), nullable=True),
    StructField("QUANTITYORDERED", IntegerType(), nullable=True),
    StructField("PRICEEACH", DecimalType(), nullable=True),
    StructField("ORDERLINENUMBER", IntegerType(), nullable=True),
    StructField("SALES", DecimalType(), nullable=True),
    StructField("ORDERDATE", StringType(), nullable=True),
    StructField("STATUS", StringType(), nullable=True),
    StructField("QTR_ID", ByteType(), nullable=True),
    StructField("MONTH_ID", ByteType(), nullable=True),
    StructField("YEAR_ID", IntegerType(), nullable=True),
    StructField("PRODUCTLINE", StringType(), nullable=True),
    StructField("MSRP", IntegerType(), nullable=True),
    StructField("PRODUCTCODE", StringType(), nullable=True),
    StructField("CUSTOMERNAME", StringType(), nullable=True),
    StructField("PHONE", StringType(), nullable=True),
    StructField("ADDRESSLINE1", StringType(), nullable=True),
    StructField("ADDRESSLINE2", StringType(), nullable=True),
    StructField("CITY", StringType(), nullable=True),
    StructField("STATE", StringType(), nullable=True),
    StructField("POSTALCODE", StringType(), nullable=True),
    StructField("COUNTRY", StringType(), nullable=True),
    StructField("TERRITORY", StringType(), nullable=True),
    StructField("CONTACTLASTNAME", StringType(), nullable=True),
    StructField("CONTACTFIRSTNAME", StringType(), nullable=True)
])

In [3]:
# 1.读取文件生成RDD，不要自动推断schema，先按文本读取
saleRDD = sc.textFile("../data/sales_data.csv")
# 去除表头第一行
header = saleRDD.first()
saleRDD = saleRDD.filter(lambda line: line != header) \
    .map(lambda line: line.split(","))

sale_Rows = saleRDD.map(lambda line: Row(
    ORDERNUMBER=int(line[0]),
    QUANTITYORDERED=int(line[1]),
    PRICEEACH=Decimal(line[2]),
    ORDERLINENUMBER=int(line[3]),
    SALES=Decimal(line[4]),
    ORDERDATE=line[5],
    STATUS=line[6],
    QTR_ID=int(line[7]),
    MONTH_ID=int(line[8]),
    YEAR_ID=int(line[9]),
    PRODUCTLINE=line[10],
    MSRP=int(line[11]),
    PRODUCTCODE=line[12],
    CUSTOMERNAME=line[13],
    PHONE=line[14],
    ADDRESSLINE1=line[15],
    ADDRESSLINE2=line[16],
    CITY=line[17],
    STATE=line[18],
    POSTALCODE=line[19],
    COUNTRY=line[20],
    TERRITORY=line[21],
    CONTACTLASTNAME=line[22],
    CONTACTFIRSTNAME=line[23]
))

df = spark.createDataFrame(sale_Rows, schema=schema)

df.printSchema()
df.show(5)

df.registerTempTable("sale_table")
spark.sql("SELECT ORDERNUMBER, QUANTITYORDERED FROM sale_table LIMIT 5").show()

spark.sql("SELECT COUNT(*) AS datacount FROM sale_table").show()



root
 |-- ORDERNUMBER: integer (nullable = true)
 |-- QUANTITYORDERED: integer (nullable = true)
 |-- PRICEEACH: decimal(10,0) (nullable = true)
 |-- ORDERLINENUMBER: integer (nullable = true)
 |-- SALES: decimal(10,0) (nullable = true)
 |-- ORDERDATE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- QTR_ID: byte (nullable = true)
 |-- MONTH_ID: byte (nullable = true)
 |-- YEAR_ID: integer (nullable = true)
 |-- PRODUCTLINE: string (nullable = true)
 |-- MSRP: integer (nullable = true)
 |-- PRODUCTCODE: string (nullable = true)
 |-- CUSTOMERNAME: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- ADDRESSLINE1: string (nullable = true)
 |-- ADDRESSLINE2: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- POSTALCODE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TERRITORY: string (nullable = true)
 |-- CONTACTLASTNAME: string (nullable = true)
 |-- CONTACTFIRSTNAME: string (nulla

D:\MINBOOK\Lib\site-packages\pyspark\sql\dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+-----------+---------------+
|ORDERNUMBER|QUANTITYORDERED|
+-----------+---------------+
|      10107|             30|
|      10121|             34|
|      10134|             41|
|      10145|             45|
|      10159|             49|
+-----------+---------------+

+---------+
|datacount|
+---------+
|        5|
+---------+



In [4]:
spark.sql("SELECT COUNT(*) AS datacount FROM sale_table").show()
df.select("ORDERNUMBER", "PRODUCTCODE", (2018 - df.YEAR_ID)).show(5)
spark.sql("select ORDERNUMBER, PRODUCTCODE, (2018 - YEAR_ID) from sale_table").show(5)


+---------+
|datacount|
+---------+
|        5|
+---------+

+-----------+-----------+----------------+
|ORDERNUMBER|PRODUCTCODE|(2018 - YEAR_ID)|
+-----------+-----------+----------------+
|      10107| "S10_1678"|              15|
|      10121| "S10_1678"|              15|
|      10134| "S10_1678"|              15|
|      10145| "S10_1678"|              15|
|      10159| "S10_1678"|              15|
+-----------+-----------+----------------+

+-----------+-----------+----------------+
|ORDERNUMBER|PRODUCTCODE|(2018 - YEAR_ID)|
+-----------+-----------+----------------+
|      10107| "S10_1678"|              15|
|      10121| "S10_1678"|              15|
|      10134| "S10_1678"|              15|
|      10145| "S10_1678"|              15|
|      10159| "S10_1678"|              15|
+-----------+-----------+----------------+



In [6]:
df.select("PRODUCTCODE").groupby("PRODUCTCODE").count().show()

+-----------+-----+
|PRODUCTCODE|count|
+-----------+-----+
| "S10_1678"|    5|
+-----------+-----+



In [8]:
spark.sql(" select PRODUCTCODE ,count(*) counts"
          " from sale_table group by PRODUCTCODE").show()

+-----------+------+
|PRODUCTCODE|counts|
+-----------+------+
| "S10_1678"|     5|
+-----------+------+

